In [1]:
import numpy as np
from sympy import *

In [2]:
# Defini os parametros
sig = 0.001
alp = 0.0001
gama = 0.5
the = 0.000001
beta = 0.001
M = 10000
epi = 0.1

## Função gradiente e hessiano

In [3]:
# Definindo as funções
def gradient_op(variables, f):
    return Matrix([Lambda(variables, f.diff(v)) for v in variables])

def hessiano_op(variables, f):
    return Matrix([[Lambda(variables, f.diff(v,u)) for u in variables] for v in variables])

In [4]:
# Funções para o cálculo
def gradient_res(grad_f, vetor):
    results = []
    for dx in grad_f:
        results.append(dx(*vetor))
    return results

def hessiano_res(hes_f, vetor):
    matrix = []
    lin = []
    arr = np.array(hes_f)
    for i in arr:
        for j in i:
            lin.append(j(*vetor))
        matrix.append(lin)
        lin = []
    return Matrix(matrix)

In [5]:
# Função para o cálculo de módulos
def modulo(x): 
    return float(sqrt(sum(i**2 for i in x)))

### Exemplos

In [6]:
# Defini variaveis
init_printing()
variables = var('x1:3')

In [7]:
# Defini a função
f = x1**4+2*x2*x1**2+x2**2-4*x1
fl = Lambda(variables, f)
fl

In [8]:
grad = gradient_op(variables, f)
grad

⎡               3              ⎤
⎢(x₁, x₂) ↦ 4⋅x₁  + 4⋅x₁⋅x₂ - 4⎥
⎢                              ⎥
⎢                  2           ⎥
⎣   (x₁, x₂) ↦ 2⋅x₁  + 2⋅x₂    ⎦

In [9]:
hes = hessiano_op(variables, f)
hes

⎡             ⎛    2     ⎞                 ⎤
⎢(x₁, x₂) ↦ 4⋅⎝3⋅x₁  + x₂⎠  (x₁, x₂) ↦ 4⋅x₁⎥
⎢                                          ⎥
⎣     (x₁, x₂) ↦ 4⋅x₁        (x₁, x₂) ↦ 2  ⎦

## Newton Globalizado

In [10]:
# Cálculo da direção
def direcao_NG(variables, beta, grad_l, hes_l):
    mi = 0
    A = []
    while True:
        A = hes_l + mi*np.identity(len(variables))
        try:
            d = np.linalg.solve(A, -grad_l)
        
            if sum(d*grad_l) > -the*modulo(grad_l)*modulo(d):
                mi = max(2*mi, beta)
            else:
                break
        except:
            mi = max(2*mi, beta)
    return d

In [11]:
# Função que aplica o método
def NG(variables, f, x, epi):
    fl = Lambda(variables, f)
    grad = gradient_op(variables, f)
    hes = hessiano_op(variables, f)
    
    k = 0
    x = x0
    while modulo(gradient_res(grad, x)) > epi and k < M:
        grad_l = np.array(gradient_res(grad, x), dtype=np.float)
        hes_l = np.array(hessiano_res(hes, x), dtype=np.float)
    
        d = direcao_NG(variables, beta, grad_l, hes_l)
            
        #fazendo as verificações de armijo
        if modulo(d) < sig*modulo(grad_l):
            d = the*(modulo(grad_l)/modulo(d))*d
        
        #calculando xk
        t = 1
        while fl(*(x+d*t)) > fl(*x) + alp*t*sum(d*grad_l):
            t = gama*t
    
        x = x+d*t
        k = k + 1
    return x, fl(*x), k

### Exemplo com NG

In [12]:
# Defini x0
x0 = [1,0]

In [13]:
# Aplica o Newton Generalizado
NG(variables, f, x0, 0.1)

(array([    964.30419384, -929881.88302472]), -3856.73315429688, 10000)

#### Função Quadrática com NG

In [72]:
init_printing()
variables = var('x1:6')
f = x1**2+2*x2**2+3*x3**2+4*x4**2+5*x5**2
fl = Lambda(variables, f)
fl

In [75]:
# Defini x0
x0 = [1,1,1,1,1]

In [76]:
# Aplica o Newton Generalizado
NG(variables, f, x0, 0.1)

(array([0., 0., 0., 0., 0.]), 0, 1)

#### Rosenbrook com NG

In [78]:
init_printing()
variables = var('x1:5')
f = 100*(x2-x1**2)**2+(x1-1)**2+100*(x4-x3**2)**2+(x3-1)**2
fl = Lambda(variables, f)
fl

In [79]:
# Defini x0
x0 = [0,0,0,0]

In [85]:
# Aplica o Newton Generalizado
NG(variables, f, x0, 0.01)

(array([0.99995223, 0.99990215, 0.99995223, 0.99990215]),
 5.63790109091734e-9,
 12)

In [87]:
NG(variables, f, x0, 0.000001)

(array([1., 1., 1., 1.]), 6.84151946004567e-28, 14)

#### Styblinsky–Tang com NG

In [96]:
init_printing()
variables = var('x1:4')
f = x1**4-16*x1**2+5*x1+x2**4-16*x2**2+5*x2+x3**4-16*x3**2+5*x3
fl = Lambda(variables, f)
fl

In [103]:
# Defini x0
x0 = [0,0,0]

In [104]:
NG(variables, f, x0, 0.01)

(array([-2.90353471, -2.90353471, -2.90353471]), -234.996994222581, 4)

#### Rastrigin com NG

In [107]:
init_printing()
variables = var('x1:4')
f = x1**2 - 10*cos(2*pi*x1)+x2**2 - 10*cos(2*pi*x2)+x3**2 - 10*cos(2*pi*x3)
fl = Lambda(variables, f)
fl

In [127]:
# Defini x0
x0 = [0,0,0]

In [133]:
NG(variables, f, x0, 0.01)

(array([4.9746914, 4.9746914, 4.9746914]),
 74.2426634715323 - 30*cos(1.94938279302003*pi),
 2)

In [134]:
30*cos(1.98991895281741*3.14)

In [135]:
# Defini x0
x0 = [5,5,5]

In [139]:
NG(variables, f, x0, 0.00001)

(array([4.9746914, 4.9746914, 4.9746914]),
 74.2426634715323 - 30*cos(1.94938279302003*pi),
 2)

In [140]:
74.2458269771614 - 30*cos(1.94959476408705*3.14)